In [4]:
from datasets import load_dataset

dataset_qna = load_dataset( 'rag-datasets/rag-mini-bioasq', name='question-answer-passages', cache_dir='./data')
dataset_corpus = load_dataset( 'rag-datasets/rag-mini-bioasq', name='text-corpus', cache_dir='./data')

In [6]:
test_qn = dataset_qna['test']
dataset_corpus['passages']

Dataset({
    features: ['passage', 'id'],
    num_rows: 40221
})

In [7]:
from pandas import DataFrame

df_qna = DataFrame(test_qn)
df_corpus = DataFrame(dataset_corpus['passages'])

In [20]:
df_corpus

,passage,id
0,New data on viruses isolated from patients wit...,9797
1,We describe an improved method for detecting d...,11906
2,We have studied the effects of curare on respo...,16083
3,Kinetic and electrophoretic properties of 230-...,23188
4,Male Wistar specific-pathogen-free rats aged 2...,23469
...,...,...
40216,LncRNAs are involved in the occurrence and pro...,34885209
40217,BACKGROUND: COVID-19 patients with long incuba...,34886835
40218,Spinal muscular atrophy (SMA) is an autosomal ...,34888619
40219,Amphiregulin (AREG) is an epidermal growth fac...,34893673


In [ ]:
import randomdef create_passage_samples(df_qna, df_corpus, n=5):    """Sample first n rows from df_qna and add relevant/non-relevant passages."""    # Takes first n rows from df_qna, looks up relevant_passage_ids in df_corpus to get passage text,    # and samples equal number of random non-relevant passages from remaining corpus.    df_qna_sampled = df_qna.head(n).copy()        # Convert corpus IDs to strings for matching    df_corpus_clean = df_corpus.copy()    df_corpus_clean['id'] = df_corpus_clean['id'].astype(str)    corpus_id_to_passage = dict(zip(df_corpus_clean['id'], df_corpus_clean['passage']))    all_corpus_ids = set(df_corpus_clean['id'])        relevant_passages = []    non_relevant_passages = []        for _, row in df_qna_sampled.iterrows():        rel_ids = row['relevant_passage_ids']        # Convert to strings for matching        if isinstance(rel_ids, list):            rel_ids = [str(rid) for rid in rel_ids]        else:            rel_ids = [str(rel_ids)]                rel_passage_list = [corpus_id_to_passage.get(rid, '') for rid in rel_ids if rid in corpus_id_to_passage]        relevant_passages.append(rel_passage_list)                num_rel = len(rel_passage_list)        non_rel_ids = list(all_corpus_ids - set(rel_ids))        sampled_non_rel = random.sample(non_rel_ids, num_rel) if num_rel > 0 else []        non_rel_passage_list = [corpus_id_to_passage.get(nid, '') for nid in sampled_non_rel]        non_relevant_passages.append(non_rel_passage_list)        df_qna_sampled['relevant_passages'] = relevant_passages    df_qna_sampled['non_relevant_passages'] = non_relevant_passages        return df_qna_sampled

In [14]:
df_sampled = create_passage_samples(df_qna, df_corpus, n=5)
df_sampled[['question', 'relevant_passages', 'non_relevant_passages']]

,question,relevant_passages,non_relevant_passages
0,Is Hirschsprung disease a mendelian or a multi...,[],[]
1,List signaling molecules (ligands) that intera...,[],[]
2,Is the protein Papilin secreted?,[],[]
3,Are long non coding RNAs spliced?,[],[]
4,Is RANKL secreted from the cells?,[],[]


In [7]:
df_sampled

,question,answer,relevant_passage_ids,id,relevant_passages,non_relevant_passages
0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 6650562, 15829955, 15617541, 230011...",0,[],[]
1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[23821377, 24323361, 23382875, 22247333, 23787...",1,[],[]
2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004...",2,[],[]
3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,"[22955974, 21622663, 22707570, 22955988, 24285...",3,[],[]
4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,"[22867712, 23827649, 21618594, 23835909, 24265...",4,[],[]


In [16]:
df_corpus[df_corpus['id'] == '23382875']

,passage,id


In [17]:
df_corpus.head()

,passage,id
0,New data on viruses isolated from patients wit...,9797
1,We describe an improved method for detecting d...,11906
2,We have studied the effects of curare on respo...,16083
3,Kinetic and electrophoretic properties of 230-...,23188
4,Male Wistar specific-pathogen-free rats aged 2...,23469
